# Trường tìm kiếm luồng issue: quét biến thể — gold 134

Luồng issue không sửa được bằng tham số (đã loại trừ: trọng số fusion, độ dài truy
vấn, chỉ-dense, chỉ-bm25). Lỗi còn lại chia gần đôi trên gold 134 — trong 38 cặp
trượt top-10: **53% không vào nổi top-100** (lỗi truy xuất), **47% có trong top-100
nhưng bị chen xuống** (lỗi xếp hạng).

`text_dense` hiện là `summary + actual + expected`. Chưa có: điều kiện kích hoạt
(`sections.steps`, `precondition`) và mã lỗi (`signals.error_code` — nay chỉ nằm
trong `text_bm25`). Mọi thứ đều có sẵn trong corpus, không gọi LLM.

## Đã chạy thử trên CPU, CHƯA có khoảng tin cậy

| biến thể | R@10 | nDCG@10 | R@100 |
|---|---|---|---|
| `base` summary+actual+expected | **0,7662** | 0,6057 | 0,8669 |
| `codes` + mã lỗi | 0,7662 | **0,6087** | 0,8632 |
| `steps` + steps | 0,7587 | 0,6081 | **0,8930** |
| `steps_precond_codes` | 0,7624 | 0,6058 | 0,8856 |
| `no_expected` bỏ expected | 0,7400 | 0,6032 | 0,8371 |
| `summary_only` chỉ summary | 0,6617 | 0,5461 | 0,8259 |
| `labelled` gắn nhãn vai trò | 0,7289 | 0,5892 | 0,8706 |

**Việc của notebook này là kiểm định, không phải khám phá.** Câu hỏi duy nhất:
`steps` nâng R@100 (+0,026) là thật hay nhiễu của 134 truy vấn?

**Đọc cột R@100 trước.** Đó là chỉ số của rổ lỗi truy xuất — rổ lớn hơn. R@10 có
thể hơi tụt mà vẫn là tin tốt: nếu `steps` lấy được ứng viên mà nền bỏ sót, phần
xếp hạng tinh để reranker lo.

Dấu `*` = khoảng tin cậy ghép cặp (bootstrap 5.000) không chứa 0.


## 1. Môi trường

`sentence-transformers` **phải ghim đúng 6.0.1**: checkpoint `e5-base-task2` được
lưu bằng bản này và tham chiếu `sentence_transformers.base.modules.normalize`, thứ
không tồn tại ở 5.x. Nới ràng buộc = `ModuleNotFoundError` lúc nạp model.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import sys; print("python", sys.version)
!pip install "sentence-transformers==6.0.1" gdown 2>&1 | tail -3

## 2. Tải bundle lên và giải nén

Chọn `colab/field_variants_bundle.zip` (6,1 MB).

In [ ]:
from google.colab import files
up = files.upload()
BUNDLE = next(iter(up))
!unzip -oq {BUNDLE} -d /content/work
%cd /content/work
!ls pipeline data/processed

## 3. Tải checkpoint e5-base-task2

Cùng file Drive đã dùng cho `rerank_window_colab.ipynb`. Luồng issue không dùng
reranker (đã tắt: rerank −0,060 nDCG@10 so với hybrid trên gold) nên chỉ cần model này.

In [ ]:
import os, glob, shutil, zipfile, gdown
FILES = {"e5-base-task2": "1rNO17nLa1kXGdmMEsUhW5Hl_9e7kjN8O"}
os.makedirs("models", exist_ok=True)
for name, fid in FILES.items():
    dst, z = "models/" + name, "/content/" + name + ".zip"
    if not os.path.exists(dst + "/config.json"):
        gdown.download(id=fid, output=z, quiet=False)
        assert zipfile.is_zipfile(z), "Khong tai duoc model ZIP: " + name
        tmp = "/content/unz_" + name
        shutil.rmtree(tmp, ignore_errors=True)
        with zipfile.ZipFile(z) as archive:
            archive.extractall(tmp)
        cfg = sorted(glob.glob(tmp + "/**/config.json", recursive=True), key=len)[0]
        shutil.move(os.path.dirname(cfg), dst)
print(sorted(os.listdir("models/e5-base-task2"))[:6])

## 4. Kiểm tra sẵn sàng — chạy ô này trước

Ô này **nạp model thật và nhúng thử hai câu**. Chỉ `import sentence_transformers`
là không đủ: bản sai vẫn import được, chỉ chết lúc nạp checkpoint. Ô này xanh thì
mọi ô sau chạy được.

In [ ]:
import os, sys
import sentence_transformers, torch
print("sentence-transformers", sentence_transformers.__version__,
      "| torch", torch.__version__, "| cuda", torch.cuda.is_available())
assert sentence_transformers.__version__.startswith("6."), (
    "Can sentence-transformers 6.x, dang co " + sentence_transformers.__version__
    + " -> chay lai o muc 1.")
assert torch.cuda.is_available(), "Khong thay GPU: Runtime > Change runtime type > T4 GPU"

for f in ("pipeline/eval_field_variants.py", "models/e5-base-task2/config.json",
          "data/processed/corpus.jsonl", "data/processed/similar_gold_eval.jsonl",
          "data/processed/pooling_judgments_v4_llm.csv"):
    assert os.path.exists(f), "THIEU: " + f
    print("co ", f)

sys.path.insert(0, "pipeline")
import eval_field_variants
print("import script OK:", len(eval_field_variants.VARIANTS), "bien the")

from sentence_transformers import SentenceTransformer
m = SentenceTransformer("models/e5-base-task2", device="cuda")
v = m.encode(["query: thu nghiem", "query: kiem tra"], batch_size=2)
print("nap model OK, vector", v.shape)
del m
torch.cuda.empty_cache()
print("SAN SANG")

## 5. Smoke test — 2 biến thể

Khoảng 1 phút. `base` phải ra đúng **R@10 0,7662 / nDCG@10 0,6057** — nếu lệch thì
môi trường sai ở đâu đó, dừng lại đừng chạy tiếp.

In [ ]:
!python pipeline/eval_field_variants.py --device cuda --batch-size 256 --variants base summary_only --n-boot 500 --out /content/smoke.json

## 6. Quét đủ 7 biến thể trên gold 134

Mỗi biến thể nhúng lại 10.925 tài liệu; trên T4 chừng 1–2 phút mỗi biến thể.

In [ ]:
!python pipeline/eval_field_variants.py --device cuda --batch-size 256 --n-boot 5000 --out data/processed/field_variants_gold.json

## 7. Chạy thêm trên pooling v4 — đọc có điều kiện

Pooling v4 **chỉ gộp top-10 của 4 cấu hình cũ**, nên tài liệu mà biến thể mới lôi
lên sẽ chưa ai chấm và bị tính grade 0. Phép đo này phạt có hệ thống mọi thay đổi
khác biệt nhất.

Dùng nó để phát hiện biến thể nào làm **tụt** rõ rệt. **Đừng** dùng để chọn biến
thể thắng, và đừng dùng R@100 ở đây — pooling không nhìn thấy gì ngoài top-10.

In [ ]:
!python pipeline/eval_field_variants.py --device cuda --batch-size 256 --pooling data/processed/pooling_judgments_v4_llm.csv --n-boot 5000 --out data/processed/field_variants_pooling.json

## 8. Tải kết quả về

Giải nén vào `D:/viettel/semantic/` — các file đã nằm sẵn đường dẫn `data/processed/`.

Nếu có biến thể thắng: **chưa đổi cấu hình pilot ngay**. Phải chấm bổ sung phần tài
liệu mới nổi lên (re-pool sâu hơn top-10) rồi mới nghiệm thu, vì lý do ở mục 7.

In [ ]:
import os
found = [f for f in ("data/processed/field_variants_gold.json",
                     "data/processed/field_variants_pooling.json")
         if os.path.exists(f)]
assert found, "Chua co ket qua nao - cuon len xem o muc 6/7 bao loi gi."
print("dong goi:", found)
!zip -q field_variants_results.zip {" ".join(found)}
!ls -lh field_variants_results.zip
from google.colab import files
files.download("field_variants_results.zip")